# 08 · Gold — Fraud Signal Summary

**Layer:** Gold &nbsp;|&nbsp; **Stage:** aggregation &nbsp;|&nbsp; **Target:** `banking_gold.fraud_signal_summary`

Second gold aggregate, in parallel with `07_gold_account_balance_daily`. Scores
each account against a set of transparent, explainable rules rather than a
model — which is what a monitoring team actually wants to see at 3am, and
what demos well because every score can be traced back to the signals that
produced it.

In the pipeline this task is deliberately placed on the **AWS cluster** while
its sibling runs on the on-prem large box, to show per-task placement across
clouds within a single DAG.

**Signals**

| Signal | Rule |
| --- | --- |
| `velocity` | 3+ transactions inside a 10-minute window |
| `high_value` | single transaction above ₹50,000 |
| `night_activity` | transactions between 22:00 and 06:00 |
| `foreign_currency` | non-INR settlement |
| `cross_border` | counterparty country is not IN |
| `structuring` | repeated amounts just under a round threshold |


In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("medallion-gold-fraud").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
SILVER_DB = "banking_silver"
GOLD_DB   = "banking_gold"

HIGH_VALUE_INR   = 50000.0
VELOCITY_WINDOW  = 600      # seconds
VELOCITY_MIN_TXN = 3

txn = spark.table(f"{SILVER_DB}.txn_clean")
print(f"scoring {txn.count()} transactions")

In [ ]:
# Velocity: how many transactions on the same account fall inside a rolling
# 10-minute window ending at each transaction.
velocity_window = (
    Window.partitionBy("account_id")
    .orderBy(F.col("txn_ts").cast("long"))
    .rangeBetween(-VELOCITY_WINDOW, 0)
)

flagged = (
    txn
    .withColumn("txns_in_window", F.count("*").over(velocity_window))
    .withColumn("sig_velocity",        F.col("txns_in_window") >= VELOCITY_MIN_TXN)
    .withColumn("sig_high_value",      F.col("amount_inr") > HIGH_VALUE_INR)
    .withColumn("sig_night_activity",  F.col("is_night_txn"))
    .withColumn("sig_foreign_currency", F.col("currency") != F.lit("INR"))
    .withColumn("sig_cross_border",    F.col("country") != F.lit("IN"))
    .withColumn(
        "sig_structuring",
        (F.col("amount_inr") % 10000 > 9000) | (F.col("amount_inr") % 10000 < 1000),
    )
)

In [ ]:
SIGNALS = {
    "sig_velocity":         30,
    "sig_high_value":       25,
    "sig_cross_border":     20,
    "sig_foreign_currency": 10,
    "sig_night_activity":   10,
    "sig_structuring":       5,
}

score_expr = sum(
    (F.when(F.col(col), F.lit(weight)).otherwise(F.lit(0)) for col, weight in SIGNALS.items()),
    F.lit(0),
)

scored = flagged.withColumn("txn_risk_score", score_expr)

scored.select(
    "txn_id", "account_id", "amount_inr", "txns_in_window",
    *SIGNALS.keys(), "txn_risk_score",
).orderBy(F.desc("txn_risk_score")).show(20, truncate=False)

In [ ]:
summary = (
    scored.groupBy("account_id", "acct_customer_id", "acct_branch_code", "acct_risk_category")
    .agg(
        F.count("*").alias("txn_count"),
        F.round(F.sum("amount_inr"), 2).alias("total_amount_inr"),
        F.round(F.max("amount_inr"), 2).alias("largest_txn_inr"),
        F.sum(F.col("sig_velocity").cast("int")).alias("velocity_hits"),
        F.sum(F.col("sig_high_value").cast("int")).alias("high_value_hits"),
        F.sum(F.col("sig_night_activity").cast("int")).alias("night_hits"),
        F.sum(F.col("sig_foreign_currency").cast("int")).alias("foreign_ccy_hits"),
        F.sum(F.col("sig_cross_border").cast("int")).alias("cross_border_hits"),
        F.sum(F.col("sig_structuring").cast("int")).alias("structuring_hits"),
        F.max("txn_risk_score").alias("peak_txn_score"),
        F.round(F.avg("txn_risk_score"), 1).alias("avg_txn_score"),
        F.collect_set("country").alias("countries_seen"),
        F.collect_set("channel").alias("channels_seen"),
    )
)

In [ ]:
fraud_signal_summary = (
    summary
    .withColumn(
        "account_risk_score",
        F.least(F.lit(100), F.round(F.col("peak_txn_score") * 0.6 + F.col("avg_txn_score") * 0.4, 0)),
    )
    .withColumn(
        "risk_tier",
        F.when(F.col("account_risk_score") >= 60, "CRITICAL")
         .when(F.col("account_risk_score") >= 40, "HIGH")
         .when(F.col("account_risk_score") >= 20, "WATCH")
         .otherwise("NORMAL"),
    )
    .withColumn("requires_review", F.col("account_risk_score") >= 40)
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .withColumn("_run_date", F.lit(RUN_DATE))
    .withColumn("_generated_at", F.current_timestamp())
)

(
    fraud_signal_summary.write
    .mode("overwrite").format("parquet")
    .saveAsTable(f"{GOLD_DB}.fraud_signal_summary")
)

out = spark.table(f"{GOLD_DB}.fraud_signal_summary")
print(f"wrote {GOLD_DB}.fraud_signal_summary  ({out.count()} rows)")

In [ ]:
out.select(
    "account_id", "acct_branch_code", "txn_count", "total_amount_inr",
    "velocity_hits", "high_value_hits", "cross_border_hits",
    "account_risk_score", "risk_tier", "requires_review",
).orderBy(F.desc("account_risk_score")).show(truncate=False)

In [ ]:
print("accounts escalated for manual review")

out.filter(F.col("requires_review")).select(
    "account_id", "acct_customer_id", "risk_tier", "account_risk_score",
    "largest_txn_inr", "countries_seen",
).orderBy(F.desc("account_risk_score")).show(truncate=False)

print("risk tier distribution")
out.groupBy("risk_tier").count().orderBy("risk_tier").show(truncate=False)